In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)


wine = pd.read_csv(r"C:\Users\ayomi\Downloads\data.csv")

print(f"Rows: {len(wine):,}")
print(f"Columns: {wine.shape[1]}")
print(f"Missing values: {wine.isna().sum().sum()}")
print(f"Exact duplicate rows: {wine.duplicated().sum()}")

print("\nQuality-score distribution:")
print(wine["quality"].value_counts().sort_index())

wine.head()


wine_clean = wine.drop_duplicates().reset_index(drop=True)

print(f"Rows before duplicate removal: {len(wine):,}")
print(f"Rows after duplicate removal:  {len(wine_clean):,}")
print(f"Rows removed: {len(wine) - len(wine_clean):,}")


wine_clean["quality_class"] = (
    wine_clean["quality"] >= 7
).astype(int)

class_summary = pd.DataFrame({
    "count": wine_clean["quality_class"]
    .value_counts()
    .sort_index()
})

class_summary.index = [
    "Bad (0)",
    "Good (1)"
]

class_summary["percent"] = (
    class_summary["count"]
    / len(wine_clean)
    * 100
).round(1)

display(class_summary)

fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    class_summary.index,
    class_summary["count"]
)

ax.set_title(
    "Wine Quality Class Distribution"
)

ax.set_ylabel(
    "Number of Wines"
)

plt.show()


X = wine_clean.drop(
    columns=[
        "quality",
        "quality_class"
    ]
)

y = wine_clean[
    "quality_class"
]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(
    "Training rows:",
    len(X_train)
)

print(
    "Test rows:",
    len(X_test)
)

print(
    "\nTraining class proportions:"
)

print(
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
)

print(
    "\nTest class proportions:"
)

print(
    y_test
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
)


dummy = DummyClassifier(
    strategy="most_frequent"
)

dummy.fit(
    X_train,
    y_train
)

dummy_pred = dummy.predict(
    X_test
)


rf = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

rf.fit(
    X_train,
    y_train
)

rf_pred = rf.predict(
    X_test
)


svc_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "svc",
        SVC(
            class_weight="balanced"
        )
    )
])


param_grid = {
    "svc__C": [
        0.1,
        0.5,
        1,
        2,
        5,
        10
    ],
    "svc__kernel": [
        "rbf"
    ],
    "svc__gamma": [
        "scale",
        0.01,
        0.05,
        0.1,
        0.2
    ]
}


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


grid_svc = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=1
)


grid_svc.fit(
    X_train,
    y_train
)


best_svc = (
    grid_svc.best_estimator_
)

svc_pred = best_svc.predict(
    X_test
)


print(
    "Best SVM parameters:",
    grid_svc.best_params_
)

print(
    f"Best cross-validation F1: "
    f"{grid_svc.best_score_:.3f}"
)


def evaluate_model(
    name,
    y_true,
    y_pred
):
    return {
        "Model": name,
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Balanced Accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),
        "Precision (Good)":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),
        "Recall (Good)":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),
        "F1 (Good)":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            )
    }


comparison = pd.DataFrame([
    evaluate_model(
        "Dummy Baseline",
        y_test,
        dummy_pred
    ),
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred
    ),
    evaluate_model(
        "Tuned SVM",
        y_test,
        svc_pred
    )
]).set_index(
    "Model"
)

display(
    comparison.round(3)
)


print(
    classification_report(
        y_test,
        svc_pred,
        target_names=[
            "Bad",
            "Good"
        ],
        zero_division=0
    )
)


fig, ax = plt.subplots(
    figsize=(6, 5)
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    svc_pred,
    display_labels=[
        "Bad",
        "Good"
    ],
    ax=ax
)

ax.set_title(
    "Tuned SVM Confusion Matrix"
)

plt.show()


fig, ax = plt.subplots(
    figsize=(6, 5)
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_pred,
    display_labels=[
        "Bad",
        "Good"
    ],
    ax=ax
)

ax.set_title(
    "Random Forest Confusion Matrix"
)

plt.show()


feature_importance = (
    pd.Series(
        rf.feature_importances_,
        index=X.columns
    )
    .sort_values(
        ascending=False
    )
)


fig, ax = plt.subplots(
    figsize=(8, 5)
)

feature_importance.sort_values().plot(
    kind="barh",
    ax=ax
)

ax.set_title(
    "Random Forest Feature Importance"
)

ax.set_xlabel(
    "Importance"
)

plt.show()


display(
    feature_importance
    .to_frame(
        "importance"
    )
    .round(3)
)
